# 40. 합성 평가셋 — 사람 검수표 만들기

사전 등록 `docs/plans/NLR_EVALSET_PREREGISTRATION.md` §9 가 요구한
**갈래별 15건씩 30건** 검수를 위한 표를 만든다.

**이 노트북은 판정하지 않는다.** 표본을 고정 시드로 뽑아 기록하고,
사람이 판정할 수 있도록 필요한 정보를 붙인 CSV 를 만든다.

## 0. 실행 조건과 한계

- **API 호출 0회.** 노트북 32·33 의 산출물만 읽는다
- **평가셋 파일을 수정하지 않는다.** 검수표는 별도 파일이다
- 표본은 **시드 42 고정**이다. 다시 돌려도 같은 30건이 나온다
- 갈래 B 의 판정 자료는 `32_evalset_input_B.csv` 의 리뷰 요약을 쓴다
  (`perfumes.jsonl` 을 다시 읽지 않는다)
- **검수 결과로 프롬프트를 고치면 사전 등록이 깨진다.** §9 참조

In [1]:
import hashlib
import pathlib

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 250)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

SEED = 42          # 표본 고정
N_PER_ARM = 15     # 사전 등록 §9

print(f"REPORT_ONLY: {REPORT_ONLY} / seed {SEED} / 갈래당 {N_PER_ARM}건")

REPORT_ONLY: False / seed 42 / 갈래당 15건


## 1. 경로 · 쓰기 가드

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"
EVAL_DIR = PROJECT_ROOT / "evaluation_data" / "nlr_evalset"

INPUT_PATHS = {
    "generated_a": EVAL_DIR / "generated_A.csv",
    "generated_b": EVAL_DIR / "generated_B.csv",
    "input_a": OUTPUT_DIR / "32_evalset_input_A.csv",
    "input_b": OUTPUT_DIR / "32_evalset_input_B.csv",
    "answer_key": OUTPUT_DIR / "32_evalset_answer_key.csv",
    "perfumes_csv": PROJECT_ROOT / "perfumes.csv",
}
OUTPUT_PATHS = {
    "review_form": OUTPUT_DIR / "40_evalset_review_form.csv",
    "sample_log": OUTPUT_DIR / "40_evalset_review_sample.json",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    d = hashlib.sha256()
    with pathlib.Path(path).open("rb") as h:
        for chunk in iter(lambda: h.read(1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
PROTECTED = {p.resolve() for p in INPUT_PATHS.values()}
ALLOWED_WRITES = {p.resolve() for p in OUTPUT_PATHS.values()}


def write_output(path, writer):
    """OUTPUT_PATHS 경로에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path not in ALLOWED_WRITES:
        raise RuntimeError(f"쓰기 허용 경로가 아닙니다: {path}")
    if path in PROTECTED:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.relative_to(PROJECT_ROOT)}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
generated_a,1ac78a370fa27581
generated_b,7c3c869594c43a2f
input_a,d2d14e131f35457b
input_b,50850ece36b7cddf
answer_key,d81e0f4fb2f1e1cd
perfumes_csv,cec1ea0b49885303


## 2. 표본 — 갈래별 15건

문장 단위로 뽑는다. 같은 향수의 3문장이 겹쳐 뽑힐 수 있고 그대로 둔다 —
**문장이 판정 단위**이기 때문이다.

In [3]:
gen = {}
for arm in ("A", "B"):
    d = pd.read_csv(INPUT_PATHS[f"generated_{arm.lower()}"])
    d["arm"] = arm
    gen[arm] = d

sample = pd.concat(
    [gen[arm].sample(n=N_PER_ARM, random_state=SEED) for arm in ("A", "B")],
    ignore_index=True)
print(f"표본 {len(sample)}건 / 갈래별 {sample.arm.value_counts().to_dict()}")
print(f"서로 다른 향수 {sample.perfume_id.nunique()}개")

표본 30건 / 갈래별 {'A': 15, 'B': 15}
서로 다른 향수 15개


## 3. 판정 자료 붙이기

사람이 *"이 문장이 그 향수를 설명하는가"* 를 판단하려면 **향수가 무엇인지** 봐야 한다.
갈래마다 생성에 쓴 자료가 다르므로 각각 붙인다.

In [4]:
meta = pd.read_csv(INPUT_PATHS["perfumes_csv"], usecols=["id", "name", "brand"],
                   low_memory=False)
key = pd.read_csv(INPUT_PATHS["answer_key"]).set_index("perfume_id")
in_a = pd.read_csv(INPUT_PATHS["input_a"]).set_index("perfume_id")["accords"]
in_b = pd.read_csv(INPUT_PATHS["input_b"]).set_index("perfume_id")["opinions"]
name_map = {r["id"]: (r["name"], r["brand"]) for r in meta.to_dict("records")}

rows = []
for n, r in enumerate(sample.to_dict("records"), 1):
    pid = r["perfume_id"]
    nm, br = name_map.get(pid, ("?", "?"))
    src = in_a.get(pid, "") if r["arm"] == "A" else in_b.get(pid, "")
    rows.append({
        "번호": n,
        "갈래": r["arm"],
        "문장": r["sentence"],
        "향수": f"{br} / {nm}",
        "채점 조건 C": str(key.loc[pid, "C"]).replace("|", ", "),
        "생성에 쓴 자료": str(src)[:180],
        "perfume_id": pid,
        "sentence_no": r["sentence_no"],
        "어색한가_YN": "",
        "규칙위반_YN": "",
        "설명이안됨_YN": "",
        "메모": "",
    })
form = pd.DataFrame(rows)
display(form.head(3)[["번호", "갈래", "문장", "향수"]])

,번호,갈래,문장,향수
0,1,A,차분한 라벤더에 바닐라 같은 달콤함이 섞인 향을 찾고 있어. 너무 달지 않게 은은한 매콤함도 있으면 해.,Tom Ford / Lavender Extreme
1,2,A,시원한 바다 느낌이 중심이지만 차갑기만 하진 않았으면 해. 나무와 따뜻한 꽃 향이 같이 나면 좋겠어.,Kenzo / Kenzo Homme Marine
2,3,A,포근하고 차분한 향이면 좋겠어. 보송한 느낌이 중심이고 나무와 살냄새가 은근하게 오래 남았으면 해.,Initio Parfums Prives / Blessed Baraka


## 4. 판정 기준 — 표에 같이 넣는다

세 칸은 **문제가 있을 때만 `Y`** 를 적는다. 비워두면 문제 없음이다.

In [5]:
CRITERIA = {
    "어색한가_YN": "한국어가 어색하거나 사람이 쓸 법하지 않은 문장이면 Y",
    "규칙위반_YN": "영어 향 용어(soapy·woody 등)나 향수 이름·브랜드·조향사·연도가 들어 있으면 Y",
    "설명이안됨_YN": "'생성에 쓴 자료'를 보고, 그 향수를 찾는 사람이 쓸 문장으로 안 읽히면 Y",
}
display(pd.Series(CRITERIA, name="판정 기준").to_frame())
print("\n빈칸 = 문제 없음. 애매하면 메모에 적고 비워둔다.")

,판정 기준
어색한가_YN,한국어가 어색하거나 사람이 쓸 법하지 않은 문장이면 Y
규칙위반_YN,영어 향 용어(soapy·woody 등)나 향수 이름·브랜드·조향사·연도가 들어 있으면 Y
설명이안됨_YN,"'생성에 쓴 자료'를 보고, 그 향수를 찾는 사람이 쓸 문장으로 안 읽히면 Y"



빈칸 = 문제 없음. 애매하면 메모에 적고 비워둔다.


## 5. 저장

In [6]:
import json

sample_log = {
    "seed": SEED,
    "n_per_arm": N_PER_ARM,
    "criteria": CRITERIA,
    "input_sha256": input_hashes_before,
    "sampled": [{"arm": r["갈래"], "perfume_id": int(r["perfume_id"]),
                 "sentence_no": int(r["sentence_no"])} for r in rows],
}
write_output(OUTPUT_PATHS["review_form"],
             lambda p: form.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["sample_log"],
             lambda p: p.write_text(json.dumps(sample_log, ensure_ascii=False, indent=2),
                                    encoding="utf-8"))

저장: analysis_outputs\40_evalset_review_form.csv
저장: analysis_outputs\40_evalset_review_sample.json


WindowsPath('C:/Users/SSAFY/Desktop/hyanghae/EDA/analysis_outputs/40_evalset_review_sample.json')

## 6. 검수 뒤에 할 일

1. `40_evalset_review_form.csv` 의 세 칸을 채운다 (문제 있으면 `Y`)
2. 결과를 `32_evalset_run_log.md` 의 검수 표에 옮긴다
3. **문제가 나오면** — 사전 등록 §9 대로 프롬프트를 고치려면 버전을 올리고 600문장을
   **전부 다시 생성**해야 한다. 몇 건을 골라 고치는 것은 허용되지 않는다

In [7]:
after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
ch = [k for k in after if after[k] != input_hashes_before[k]]
if ch:
    raise RuntimeError(f"입력 파일이 변경됐다: {ch}")
print("입력 해시 불변 확인:", ", ".join(INPUT_PATHS))
print()
print("다음 — analysis_outputs/40_evalset_review_form.csv 를 열어 30건을 판정한다.")

입력 해시 불변 확인: generated_a, generated_b, input_a, input_b, answer_key, perfumes_csv

다음 — analysis_outputs/40_evalset_review_form.csv 를 열어 30건을 판정한다.
